# Lab 01 solution: stream and price a turn, and keep a conversation within a token budget

The two challenge tasks of `01_model_call.ipynb`, solved and run. This notebook runs on its own: the setup repeats
what the tasks use from the lab (the client, the two prices of Part F and the support desk conversation of Part E).

## Setup

The same first cell as the lab, then the client and the prices.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI()

# Dollars per 1 million tokens, from models.env
PRICE_INPUT = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUTPUT = float(settings["PRICE_OUTPUT_PER_MILLION"])
print("input:", PRICE_INPUT, "| output:", PRICE_OUTPUT)

input: 0.1 | output: 0.5


Part E's conversation: the desk's system message, the customer's first message, the model's reply to it, and the
request for a note. As in the lab, the reply to turn 1 goes back into the list as an `assistant` message.

In [3]:
DESK = {
    "role": "system",
    "content": "You are a support assistant for an online music shop. Answer in one or two short sentences.",
}
customer_1 = {"role": "user", "content": "Hi, I was charged twice for my order, number 46."}
customer_2 = {"role": "user", "content": "Please write a one-line note about this for your billing team."}

desk_first = client.chat.completions.create(
    model=MODEL,
    messages=[DESK, customer_1],
    reasoning_effort=EFFORT or omit,
)
desk_history = [
    DESK,
    customer_1,
    {"role": "assistant", "content": desk_first.choices[0].message.content},
    customer_2,
]

print(desk_first.choices[0].message.content)

I’m sorry about that. I can’t access billing records here, but please contact our support team with order #46 and both charge dates and amounts so they can verify the duplicate charge and arrange a refund if needed.


## Task 1: Stream the note for the billing team, and price that call

Part C's stream, sent with `desk_history` instead of the joke. The loop keeps the `chunk.choices` check, because
the last chunk, the one that carries the usage, has an empty `choices` list.

In [4]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=desk_history,
    reasoning_effort=EFFORT or omit,
    stream=True,
    stream_options={"include_usage": True},
)

pieces = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        piece = chunk.choices[0].delta.content
        print(piece, end="|")
        pieces.append(piece)
    last_chunk = chunk

Please| review| order| #|46|:| the| customer| reports| being| charged| twice| and| requests| verification| and| a| refund| if| the| duplicate| is|

 confirmed|.|

The note arrives in 24 pieces and names order #46. The order number is two pieces, `#` and `46`, and each common
word is one piece with its space in front, as in Part C.

The price is Part F's formula on one call: the last chunk's `prompt_tokens` at the input price and its
`completion_tokens` at the output price.

In [5]:
usage = last_chunk.usage
cost = (usage.prompt_tokens * PRICE_INPUT + usage.completion_tokens * PRICE_OUTPUT) / 1_000_000

print("note:", "".join(pieces))
print("usage:", usage.prompt_tokens, "prompt +", usage.completion_tokens, "completion tokens")
print(f"price: ${cost:.7f}")

note: Please review order #46: the customer reports being charged twice and requests verification and a refund if the duplicate is confirmed.
usage: 110 prompt + 27 completion tokens
price: $0.0000245


The stream's last chunk reports 110 prompt tokens and 27 completion tokens, so the call cost \$0.0000245: 110 ×
0.10 + 27 × 0.50, per million. In the lab `desk_second` has the same `prompt_tokens` as your stream, because both
send `desk_history`. Here the number differs from the lab's saved 106 because this notebook made its own first
call, and its reply, which sits in the history, is a little longer. Streaming changed when the words arrived, not
the bill.

## Task 2: Keep the two-turn conversation within a token budget

The two turns are run three times here, with different settings, so the same steps go into a function: turn 1
alone, then turn 2 with the history, as in Part E. It returns what the check needs: the two calls' `total_tokens`
added up, their two `finish_reason` values, and the note. `**settings_for_call` passes any extra argument, such
as a ceiling, on to both calls.

In [6]:
BUDGET = 190


def two_turns(system_text, **settings_for_call):
    system = {"role": "system", "content": system_text}
    one = client.chat.completions.create(
        model=MODEL, messages=[system, customer_1], reasoning_effort=EFFORT or omit, **settings_for_call
    )
    history = [system, customer_1, {"role": "assistant", "content": one.choices[0].message.content}, customer_2]
    two = client.chat.completions.create(
        model=MODEL, messages=history, reasoning_effort=EFFORT or omit, **settings_for_call
    )
    total = one.usage.total_tokens + two.usage.total_tokens
    finishes = (one.choices[0].finish_reason, two.choices[0].finish_reason)
    print("turn 1:", one.choices[0].message.content)
    print("turn 2:", two.choices[0].message.content)
    return total, finishes, two.choices[0].message.content

The check reads three things: the total against the budget, both `finish_reason` values (a reply that was cut to
save tokens fails, as in Part B), and `46` in the note. A pass says the conversation fit and the note still names
the order. It does not say the first reply was helpful.

In the cells below, `*two_turns(...)` unpacks the three values the function returns into `check`'s last three
arguments.

In [7]:
def check(label, total, finishes, note):
    passed = total <= BUDGET and finishes == ("stop", "stop") and "46" in note
    mark = "✅" if passed else "❌"
    print(f"{mark} {label}: {total} tokens (budget {BUDGET}), finish_reason {finishes}, names 46: {'46' in note}")

First, Part E's system message as it is.

In [8]:
check("Part E's system message", *two_turns(DESK["content"]))

turn 1: I’m sorry about that. I can’t access billing records here, but please contact our support team with order #46 and both charge dates/amounts so they can verify the duplicate and arrange a refund if needed.
turn 2: Order #46: Customer reports being charged twice; please review the billing records and refund any duplicate charge.
❌ Part E's system message: 224 tokens (budget 190), finish_reason ('stop', 'stop'), names 46: True


224 tokens against a budget of 190: both replies ended with `stop` and the note names order 46, but the
conversation does not fit. Over the runs we made, this system message gave between about 200 and 230 tokens.

The first reply is paid for twice: once as output, and again as input when it is sent back in the history. So the
place to save is the length of the replies. This system message asks for one sentence of at most 12 words.

In [9]:
SHORT = "You are a support assistant for an online music shop. Answer in one sentence of at most 12 words."

check("a system message that asks for less", *two_turns(SHORT))

turn 1: Sorry about that—please contact support with order 46 so we can investigate and arrange a refund.
turn 2: Billing team: Customer reports being charged twice for order 46; please investigate and refund any duplicate charge.
✅ a system message that asks for less: 182 tokens (budget 190), finish_reason ('stop', 'stop'), names 46: True


182 tokens, both replies complete, and the note still names order 46. The first reply shrank to one sentence,
and that saving counts twice, because the reply is also part of the second call's prompt.

The other idea is to leave the system message alone and lower the ceiling. With `max_completion_tokens=15`:

In [10]:
check("a ceiling of 15 tokens", *two_turns(DESK["content"], max_completion_tokens=15))

turn 1: Sorry about that. I can’t access billing records here,
turn 2: Please investigate order 46: the customer reports being charged twice
❌ a ceiling of 15 tokens: 151 tokens (budget 190), finish_reason ('length', 'length'), names 46: True


151 tokens, well inside the budget, and the check still fails: both replies ended with `length`. The first one
stops after "here," and the note stops after "charged twice", without a full stop. The total is lower only
because the replies were cut, and `finish_reason` is what tells a cut reply from a short one.

What to take from the three runs: a token budget is met by asking for less, in the system message, and it is
checked with `finish_reason`, so that a reply cut in the middle never counts as a saving.